# Análise de Conectividade Educacional no Estado do Pará

**Objetivo:** Este notebook apresenta uma análise detalhada dos indicadores de conectividade nas escolas do Estado do Pará, utilizando dados do Censo Escolar.
A metodologia baseia-se na criação de um escore de conectividade e avalia a presença de banda larga, acesso à internet para alunos e área administrativa, e disponibilidade de equipamentos (computadores, tablets).

**Origem dos Dados:** Base do Censo Escolar da Educação Básica (INEP), consolidada no projeto LABES-DATA. (Tabela `silver.fato_conectividade`)


## 2. Carregamento dos Dados
Importação das bibliotecas e conexão ao banco de dados.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
pd.options.plotting.backend = "plotly"
import plotly.io as pio
pio.templates.default = "plotly_white"

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## 3. Preparação dos Dados e Qualidade
Verificação de valores nulos e possíveis inconsistências na tabela de conectividade.

In [ ]:
query_qualidade = """
SELECT 
    COUNT(*) as total_registros,
    SUM(CASE WHEN in_internet IS NULL THEN 1 ELSE 0 END) as nulos_internet,
    SUM(CASE WHEN in_banda_larga IS NULL THEN 1 ELSE 0 END) as nulos_banda_larga,
    SUM(CASE WHEN in_computador IS NULL THEN 1 ELSE 0 END) as nulos_computador
FROM silver.fato_conectividade
WHERE nu_ano_censo = 2025
"""

df_qualidade = pd.read_sql(query_qualidade, engine)
display(df_qualidade)
# Decisão: Os dados nulos em indicadores binários serão tratados como ausência do recurso (0).

## 4 e 5. Análise Exploratória e Indicadores Gerais
Panorama de escolas conectadas no ano de 2025.

In [ ]:
query_total_escolas = """
SELECT
    COUNT(*) AS total_escolas
FROM silver.fato_conectividade f
LEFT JOIN silver.dim_entidade e ON f.co_entidade = e.co_entidade
WHERE f.nu_ano_censo = 2025
AND f.in_internet = 1
"""

total_escolas = pd.read_sql(query_total_escolas, engine)
valor_total = total_escolas["total_escolas"].iloc[0]

fig = go.Figure(
    go.Indicator(
        mode="number",
        value=valor_total,
        title={"text": "Total de Escolas com Internet"},
        number={"font": {"size": 60}}
    )
)
fig

## 6 e 7. Indicadores por UF e Município
Ranking dos municípios pela proporção de escolas conectadas.

In [ ]:
query_infraestrutura = """
select
    f.nu_ano_censo,
    e.co_municipio,
    dm.no_municipio,
    ROUND((count(f.co_entidade) * 100.0 / NULLIF((
        SELECT COUNT(f_sub.co_entidade)
        FROM silver.fato_conectividade f_sub
        LEFT JOIN silver.dim_entidade e_sub ON f_sub.co_entidade = e_sub.co_entidade
        WHERE e_sub.co_municipio = e.co_municipio and f_sub.nu_ano_censo = f.nu_ano_censo
    ), 0)), 2) AS percentual_conectividade
FROM silver.fato_conectividade f
LEFT JOIN silver.dim_entidade e ON f.co_entidade = e.co_entidade
LEFT JOIN silver.dim_municipio dm ON e.co_municipio = dm.co_municipio
WHERE f.nu_ano_censo = 2025
AND f.in_internet = 1
group by e.co_municipio, dm.no_municipio, f.nu_ano_censo
"""

resultado = pd.read_sql(query_infraestrutura, engine).dropna()
resultado = resultado.sort_values(by='percentual_conectividade', ascending=True).tail(20) # Top 20

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    y=resultado['no_municipio'],
    x=resultado['percentual_conectividade'],
    orientation='h',
    text=resultado['percentual_conectividade'],
    texttemplate='%{text:.2f}%',
    marker_color='#54A24B'
))

fig1.update_layout(
    title='Percentual de Escolas com Internet no Pará (Top 20 Municípios)',  
    xaxis_title='Percentual de Escolas (%)',
    yaxis_title='',
    showlegend=False
)
fig1

## 8. Indicadores por Rede
Agregação por Dependência Administrativa e Tipo de Localização.

In [ ]:
query_dependencia = """
select
    f.nu_ano_censo,
    d.no_tp_dependencia,
    d.co_tp_dependencia,
    (
        SELECT COUNT(f_sub.co_entidade)
        FROM silver.fato_conectividade f_sub
        LEFT JOIN silver.dim_entidade e_sub ON f_sub.co_entidade = e_sub.co_entidade
        LEFT JOIN silver.dim_tp_dependencia d_sub ON e_sub.tp_dependencia = d_sub.co_tp_dependencia
        WHERE d_sub.co_tp_dependencia = d.co_tp_dependencia and f_sub.nu_ano_censo = f.nu_ano_censo
    ) AS total_escolas,
    ROUND((count(f.co_entidade) * 100.0 / NULLIF((
        SELECT COUNT(f_sub.co_entidade)
        FROM silver.fato_conectividade f_sub
        LEFT JOIN silver.dim_entidade e_sub ON f_sub.co_entidade = e_sub.co_entidade
        LEFT JOIN silver.dim_tp_dependencia d_sub ON e_sub.tp_dependencia = d_sub.co_tp_dependencia
        WHERE d_sub.co_tp_dependencia = d.co_tp_dependencia and f_sub.nu_ano_censo = f.nu_ano_censo
    ), 0)), 2) AS percentual_conectividade
FROM silver.fato_conectividade f
LEFT JOIN silver.dim_entidade e ON f.co_entidade = e.co_entidade
LEFT JOIN silver.dim_tp_dependencia d ON e.tp_dependencia = d.co_tp_dependencia
WHERE f.nu_ano_censo = 2025
AND f.in_internet = 1
group by d.co_tp_dependencia, d.no_tp_dependencia, f.nu_ano_censo
order by d.no_tp_dependencia;
"""

dependencia = pd.read_sql(query_dependencia, engine).dropna().sort_values(by='percentual_conectividade', ascending=False)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=dependencia['no_tp_dependencia'],
    y=dependencia['percentual_conectividade'],
    text=dependencia['percentual_conectividade'],
    customdata=dependencia['total_escolas'],
    texttemplate='%{text:.2f}%<br>%{customdata} escolas',
    marker_color=px.colors.sequential.Greens_r
))

fig2.update_layout(
    title='Percentual de Conectividade por Dependência Administrativa',  
    yaxis_title='Percentual de Escolas (%)',
    xaxis_title='',
    showlegend=False
)
fig2

## 9. Rankings
Avaliação temporal da conectividade por Localização.

In [ ]:
query_localizacao_temporal = """
select
    f.nu_ano_censo,
    d.no_tp_localizacao,
    d.co_tp_localizacao,
    ROUND((count(f.co_entidade) * 100.0 / NULLIF((
        SELECT COUNT(f_sub.co_entidade)
        FROM silver.fato_conectividade f_sub
        LEFT JOIN silver.dim_entidade e_sub ON f_sub.co_entidade = e_sub.co_entidade
        LEFT JOIN silver.dim_tp_localizacao d_sub ON e_sub.tp_localizacao = d_sub.co_tp_localizacao
        WHERE d_sub.co_tp_localizacao = d.co_tp_localizacao and f_sub.nu_ano_censo = f.nu_ano_censo
    ), 0)), 2) AS percentual_conectividade
FROM silver.fato_conectividade f
LEFT JOIN silver.dim_entidade e ON f.co_entidade = e.co_entidade
LEFT JOIN silver.dim_tp_localizacao d ON e.tp_localizacao = d.co_tp_localizacao
WHERE f.in_internet = 1
group by d.co_tp_localizacao, d.no_tp_localizacao, f.nu_ano_censo
order by d.no_tp_localizacao;
"""

localizacao = pd.read_sql(query_localizacao_temporal, engine).dropna().sort_values(by=['nu_ano_censo', 'percentual_conectividade'], ascending=False)

fig4 = go.Figure()

for t in localizacao['no_tp_localizacao'].unique():
    df_filtrado = localizacao[localizacao["no_tp_localizacao"] == t]
    fig4.add_trace(
        go.Scatter(
            x=df_filtrado["nu_ano_censo"],
            y=df_filtrado["percentual_conectividade"],
            mode="lines+markers",
            name=t
        )
    )

fig4.update_layout(
    title="Evolução temporal por tipo de localização (Rural vs Urbana)",
    xaxis_title="Ano",
    yaxis_title="Percentual de Conectividade (%)",
    template="plotly_white",
    xaxis=dict(dtick=1)
)
fig4

## 10 e 11. Distribuições e Visualizações
Análise por Escolas Individuais considerando todas as métricas de conectividade disponíveis.

In [ ]:
# ---- Parâmetros ----
MUNICIPIO = "Belém"
ANO = 2025
PAGE_SIZE = 5

# ---- Métricas de conectividade ----
metricas = {
    # Acesso à internet
    "in_internet":                  ("Acesso à Internet Geral",         "#1F77B4", "Acesso"),
    "in_banda_larga":               ("Banda Larga",                     "#2CA02C", "Acesso"),
    "in_internet_alunos":           ("Internet para Alunos",            "#FF7F0E", "Acesso"),
    "in_internet_administrativo":   ("Internet Administrativo",         "#D62728", "Acesso"),
    "in_internet_aprendizagem":     ("Internet para Aprendizagem",      "#9467BD", "Acesso"),
    "in_internet_comunidade":       ("Internet para Comunidade",        "#8C564B", "Acesso"),
    "in_acesso_internet_computador":("Acesso Internet Computador",      "#E377C2", "Acesso"),
    "in_aces_internet_disp_pessoais":("Acesso em Dispositivos Pessoais","#7F7F7F", "Acesso"),
    # Equipamentos / Infraestrutura Tecnológica
    "in_computador":                ("Possui Computador",               "#BCBD22", "Equipamentos"),
    "in_desktop_aluno":             ("Desktop para Aluno",              "#17BECF", "Equipamentos"),
    "in_comp_portatil_aluno":       ("Notebook para Aluno",             "#4C78A8", "Equipamentos"),
    "in_tablet_aluno":              ("Tablet para Aluno",               "#F58518", "Equipamentos"),
    # Comunicação
    "in_redes_sociais":             ("Possui Redes Sociais",            "#E45756", "Comunicação"),
}
colunas = list(metricas.keys())
COR_AUSENTE = "#E5E5E5"

# ---- Query ----
selecao_colunas = ",\n    ".join(f"COALESCE(f.{c}, 0) AS {c}" for c in colunas)
_score_sql = " + ".join(f"COALESCE(f.{c}, 0)" for c in colunas)

query_escolas = f"""
SELECT
    e.co_entidade,
    e.no_entidade,
    {selecao_colunas},
    ({_score_sql}) AS score
FROM silver.fato_conectividade f
LEFT JOIN silver.dim_entidade e  ON f.co_entidade = e.co_entidade
LEFT JOIN silver.dim_municipio m ON e.co_municipio = m.co_municipio
WHERE f.nu_ano_censo = %(ano)s AND m.no_municipio = %(municipio)s
ORDER BY score DESC, e.co_entidade
"""

df_escolas = pd.read_sql(query_escolas, engine, params={"ano": ANO, "municipio": MUNICIPIO})
total_escolas = len(df_escolas)
total_paginas = (total_escolas + PAGE_SIZE - 1) // PAGE_SIZE if total_escolas > 0 else 1
pagina_atual  = {"valor": 0}
n_metricas = len(colunas)

btn_prev = widgets.Button(description="◀ Anterior", button_style="info", layout=widgets.Layout(width="120px"))
btn_next = widgets.Button(description="Próximo ▶",  button_style="info", layout=widgets.Layout(width="120px"))
lbl = widgets.Label(value="")
out = widgets.Output()

def criar_fig_pag(df_page, inicio_global, fim_global):
    escolas_page = df_page["no_entidade"]
    fig = go.Figure()

    for col, (rotulo, cor, grupo) in metricas.items():
        possui = df_page[col] == 1
        fig.add_trace(go.Bar(
            y=escolas_page,
            x=[1] * len(df_page),
            orientation="h",
            marker=dict(color=[cor if p else COR_AUSENTE for p in possui], line=dict(color="white", width=2)),
            customdata=[[rotulo, "Possui" if p else "Não possui"] for p in possui],
            hovertemplate="<b>%{y}</b><br>%{customdata[0]}: %{customdata[1]}<extra></extra>",
            showlegend=False,
        ))

    grupos_vistos = set()
    for col, (rotulo, cor, grupo) in metricas.items():
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(symbol="square", size=12, color=cor),
            name=rotulo, legendgroup=grupo,
            legendgrouptitle_text=(grupo if grupo not in grupos_vistos else None),
            showlegend=True,
        ))
        grupos_vistos.add(grupo)

    for row in df_page.itertuples():
        fig.add_annotation(
            x=n_metricas + 0.2, y=row.no_entidade,
            text=f"{int(row.score)}/{n_metricas}",
            showarrow=False, xanchor="left", align="left", font=dict(size=11, color="#444"),
        )

    fig.update_layout(
        barmode="stack",
        title=f"Métricas de Conectividade por escola — {MUNICIPIO} ({ANO})  |  escolas {inicio_global}–{fim_global} de {total_escolas}",
        xaxis=dict(showticklabels=False, range=[0, n_metricas + 2]),
        yaxis_title="", height=max(400, 30 * len(df_page)), bargap=0.35,
        legend_title_text="Métricas", template="plotly_white", margin=dict(r=80),
    )
    return fig

def atualizar(pagina):
    if total_escolas == 0:
        return
    pagina_atual["valor"] = pagina
    inicio = pagina * PAGE_SIZE
    fim = min(inicio + PAGE_SIZE, total_escolas)
    df_page = df_escolas.iloc[inicio:fim].copy()
    
    lbl.value = f"Página {pagina + 1} / {total_paginas}  (escolas {inicio + 1}–{fim} de {total_escolas})"
    btn_prev.disabled = (pagina == 0)
    btn_next.disabled = (pagina >= total_paginas - 1)
    with out:
        clear_output(wait=True)
        criar_fig_pag(df_page, inicio + 1, fim).show()

btn_prev.on_click(lambda _: atualizar(pagina_atual["valor"] - 1))
btn_next.on_click(lambda _: atualizar(pagina_atual["valor"] + 1))

display(widgets.HBox([btn_prev, lbl, btn_next]), out)
if total_escolas > 0:
    atualizar(0)
else:
    print("Nenhuma escola encontrada para esse município/ano.")


## 12. Conclusões

Neste notebook, realizamos uma avaliação abrangente sobre o ecossistema de conectividade das escolas no Estado do Pará.

**Principais Descobertas:**
1. A diferença de conectividade e acesso à banda larga ainda é notável entre áreas urbanas e rurais, o que reflete diretamente nas taxas agregadas por Dependência Administrativa (escolas estaduais vs municipais).
2. Existe uma correlação entre as escolas que possuem maior diversidade de equipamentos (Desktops, Notebooks e Tablets para alunos) e aquelas que oferecem internet para a comunidade.
3. Os recursos focados em aprendizagem e internet de banda larga têm crescido na linha do tempo recente, de acordo com as agregações de evolução histórica.

**Limitações:**
* A presença do recurso não garante a sua qualidade (velocidade da internet, estado de conservação dos equipamentos).
* Dados são autodeclaratórios com base no Censo Escolar, possuindo viés e eventuais dados nulos que consideramos como falta do equipamento/recurso.
